# Lilylet patchifier visualization

This notebook loads `demo-BWV610.lyl`, converts it to NotaGen-style Lilylet patches with `starry.lilylet.data.patchifier`, and prints a readable view of the resulting patch grid.

In [2]:
from pathlib import Path
import sys

# Notebook is stored under deep-starry/tests. Add repo root to sys.path.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from starry.lilylet.data.patchifier import (
    LilyletTokenizer,
    patchify_text,
    split_lilylet_document,
    split_measures,
)

TOKENIZER_PATH = REPO_ROOT / 'assets' / 'manual-tokenizer.json'

print('repo:', REPO_ROOT)
print('tokenizer:', TOKENIZER_PATH)

repo: /home/camus/work/deep-starry
tokenizer: /home/camus/work/deep-starry/assets/manual-tokenizer.json


In [3]:
text = r'''[composer "Schubert, Franz"]
[genre "Classical"]
[instrument "Art Song"]

\staff "1" \key b \major \time 3/4 \clef "treble" \tempo 4=100 r8 \\\
\staff "1" d'8\p( \\
\staff "2" \clef "bass" r8 | %1

\staff "1" \key b \major \time 3/4 r2. \\\
\staff "1" b'8 r16 a( g4. f8)( \\
\staff "2" r8 r16 <d, c'>( <g b>4. <d c'>8)(( | %2
'''

tokenizer = LilyletTokenizer(str(TOKENIZER_PATH))

metadata_lines, body_lines = split_lilylet_document(text)
measures = split_measures(body_lines)

print(f'input chars: {len(text)}')
print(f'metadata lines: {len(metadata_lines)}')
print(f'body lines: {len(body_lines)}')
print(f'measures: {len(measures)}')
print('--- metadata ---')
print(''.join(metadata_lines) or '(none)')
print('--- first measure ---')
print(measures[0] if measures else '(none)')

input chars: 326
metadata lines: 3
body lines: 6
measures: 2
--- metadata ---
[composer "Schubert, Franz"]
[genre "Classical"]
[instrument "Art Song"]

--- first measure ---
\staff "1" \key b \major \time 3/4 \clef "treble" \tempo 4=100 r8 \\\
\staff "1" d'8\p( \\
\staff "2" \clef "bass" r8 |



In [4]:
PATCH_SIZE = 16
PATCH_LENGTH = 2048
PATCH_STREAM = True

patches, mask, unknowns = patchify_text(
    text,
    tokenizer,
    file='demo-BWV610.lyl',
    patch_size=PATCH_SIZE,
    patch_length=PATCH_LENGTH,
    patch_stream=PATCH_STREAM,
)

print('patches shape:', tuple(patches.shape))
print('mask shape:', tuple(mask.shape))
print('real patch count:', int(mask.sum()))
print('unknown total:', sum(hit['count'] for hit in unknowns))
unknowns[:10]

patches shape: (18, 16)
mask shape: (18,)
real patch count: 18
unknown total: 0


[]

In [5]:
id_to_token = {entry['id']: entry['token'] for entry in tokenizer.vocab}
special_ids = {tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id, tokenizer.unknown_id}

def display_token(token_id: int) -> str:
    token = id_to_token.get(token_id)
    if token is None:
        return f'<id:{token_id}>'
    if token == '\n':
        return r'\n'
    if token == '\t':
        return r'\t'
    if token == ' ':
        return '·'
    if token_id == tokenizer.pad_id:
        return '<pad>'
    return token

def show_patch(index: int):
    ids = [int(x) for x in patches[index].tolist()]
    rendered = [display_token(x) for x in ids]
    print(f'patch {index:04d} | mask={int(mask[index])}')
    print('ids:   ', ids)
    print('tokens:', ' | '.join(rendered))

for i in range(min(24, patches.shape[0])):
    show_patch(i)
    print('-' * 100)

patch 0000 | mask=1
ids:    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]
tokens: <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <eos>
----------------------------------------------------------------------------------------------------
patch 0001 | mask=1
ids:    [91, 160, 32, 34, 83, 99, 104, 117, 98, 101, 114, 116, 44, 32, 70, 114]
tokens: [ | composer | · | " | S | c | h | u | b | e | r | t | , | · | F | r
----------------------------------------------------------------------------------------------------
patch 0002 | mask=1
ids:    [97, 110, 122, 34, 93, 10, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]
tokens: a | n | z | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad>
----------------------------------------------------------------------------------------------------
patch 0003 | mask=1
ids:    [91, 209, 32, 34, 67, 108, 97, 255, 105, 99, 251, 34, 93, 10, 2, 0]
tokens: [ | genre

In [6]:
# Compact table view for the first N patches.
N = min(80, patches.shape[0])
for i in range(N):
    rendered = [display_token(int(x)) for x in patches[i].tolist()]
    print(f'{i:04d}  ' + ' '.join(f'{tok:>12}' for tok in rendered))

0000         <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <eos>
0001             [     composer            ·            "            S            c            h            u            b            e            r            t            ,            ·            F            r
0002             a            n            z            "            ]           \n        <eos>        <pad>        <pad>        <pad>        <pad>        <pad>        <pad>        <pad>        <pad>        <pad>
0003             [        genre            ·            "            C            l            a           ss            i            c           al            "            ]           \n        <eos>        <pad>
0004             [   instrument            ·            "            A            r            t            ·            S            o         